# Lab 13: AI APIs as Data Tools

*INFO 4617/5617 · Week 13 · Chapter 13, [AI APIs as Data Tools](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html)*

This week every class is live coding: one lab in three parts, one a day. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it.

Monday and Wednesday work on the City of Boulder's revenue reports from week 9, the PDFs whose last page is a picture of a table. Friday moves to Wikipedia:
- **Part 1, Monday:** **document understanding**, reading a whole PDF, pictures of tables included. [Datalab](https://www.datalab.to)'s API turns it into Markdown a program can read, and we check what it read;
- **Part 2, Wednesday:** **a model as a function**. Through [OpenRouter](https://openrouter.ai), a language model reads one page and returns JSON in a shape we define, and we check its numbers against week 9's;
- **Part 3, Friday:** **search by meaning**. Embeddings, lists of numbers that stand for what a passage is about, measure how the opening paragraphs of five Wikipedia articles about politicians have changed since 2004, and find the versions that answer a question. A model answers from those versions and names its sources, or says it can't answer. Last, a look at how MCP and skills describe tools to a model.

No chatbots: every call to a model is one request in and one JSON answer out, and every answer gets checked. Along the way, short ⏱ microsprints ask you to change one thing, alone or in pairs, and share what you find.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. You need **two free keys**, each stored as an environment variable by one of week 11's three safe ways (set it, close the terminal, open a new one, and start Jupyter from there):
- `DATALAB_KEY`: sign up at [datalab.to](https://www.datalab.to), then copy a key from [API Keys](https://www.datalab.to/app/keys). The free plan includes a monthly allowance of at least $10 ([pricing](https://www.datalab.to/pricing)); this lab spends about 9 cents of it;
- `OPENROUTER_KEY`: sign up at [openrouter.ai](https://openrouter.ai), then create a key at [Keys](https://openrouter.ai/settings/keys). The models we use end in `:free`: they cost nothing, up to a daily limit.

Treat each key like a password: never paste it into a cell or a code file. On Wednesday and Friday, start at that day's setup step, or run every cell above where we stopped (**Run → Run All Above Selected Cell**).

> **Instructor key.** PLACEHOLDER

# Part 1 — Monday: from a PDF to text a program can read

In week 9, pdfplumber read the text layer of Boulder's revenue reports exactly. It stopped at Exhibit 3 of the December 2024 report, a picture of a table, beside a note asking readers to phone the city for "a screen readable version". Today a **document-understanding** API, one that reads a page's pictures as well as its text, reads the whole report. Then we check what it read, the way week 9 checked pdfplumber.

## Step 0 — Set up, and check your keys (5 minutes)

Import what we need, define the User-Agent, and check that both keys are set before reading them, as week 11 did. If one says `MISSING`, quit Jupyter, open a new terminal, and start Jupyter again from there. If you can't restart now, run the given cell: it asks for a missing key with [`getpass()`](https://docs.python.org/3/library/getpass.html), which doesn't show what you type, and does nothing if both keys are set. The chapter's ["Two Keys"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#two-keys) has the steps to get them.

In [1]:
# Import os, re, time, requests, and pandas (as pd); define HEADERS as in week 12, with your email
# Print whether "DATALAB_KEY" and "OPENROUTER_KEY" are set or MISSING, without printing either key
import os
import re
import time
import requests
import pandas as pd

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
print("DATALAB_KEY:", "set" if os.environ.get("DATALAB_KEY") else "MISSING")
print("OPENROUTER_KEY:", "set" if os.environ.get("OPENROUTER_KEY") else "MISSING")

DATALAB_KEY: set
OPENROUTER_KEY: set


In [2]:
# Given: only for a key the check said was MISSING, ask for it with getpass. If both said set, this does nothing
from getpass import getpass
for name in ["DATALAB_KEY", "OPENROUTER_KEY"]:
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name} isn't set. Paste it here: ")

In [3]:
# Read DATALAB_KEY into datalab_key; datalab_base = "https://www.datalab.to/api/v1"
# book: the address of week 9's saved PDFs, "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/"
datalab_key = os.environ["DATALAB_KEY"]
datalab_base = "https://www.datalab.to/api/v1"
book = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/"


<details>
<summary>💬 Discuss</summary>

- Census, FRED and YouTube gave you keys to read their data; Datalab and OpenRouter read whatever you send them. What wouldn't you send?
- Datalab bills by the page, after a free allowance. What does that allowance let the company learn about you?

</details>

## Step 1 — Submit a document (8 minutes)

Datalab's [Convert API](https://documentation.datalab.to/docs/recipes/conversion/conversion-api-overview) takes a PDF, uploaded or by its address, and returns Markdown. We send the address of the book's saved copy of the December 2024 report as `file_url`. We ask for `markdown` in `balanced` mode, and set `paginate` so that each page is marked. Datalab keeps a copy of what it converts, and hands that copy back, free, when the same file comes again. So we also set [`skip_cache`](https://documentation.datalab.to/api-reference/convert-document): Datalab converts the file afresh, and the cost we see is the real one. The key goes in an `X-API-Key` header. The answer isn't the document. Converting takes seconds, so the API answers at once with a **ticket**, `request_check_url`: an address where the result will be when it's ready. See the chapter's ["Submitting a Document"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#submitting-a-document).

In [4]:
# data: file_url (book + "revenue-report-2024-12.pdf"), output_format "markdown", mode "balanced", paginate "true", skip_cache "true"
# job: the JSON of a POST to f"{datalab_base}/convert" with data= and headers {"X-API-Key": datalab_key} plus HEADERS; print it
data = {"file_url": book + "revenue-report-2024-12.pdf", "output_format": "markdown",
        "mode": "balanced", "paginate": "true", "skip_cache": "true"}
job = requests.post(f"{datalab_base}/convert", data=data,
                    headers={"X-API-Key": datalab_key, **HEADERS}).json()
print(job)

{'success': True, 'error': None, 'request_id': '8TiORZqhcBr2wJgqyTs-gA', 'request_check_url': 'https://www.datalab.to/api/v1/convert/8TiORZqhcBr2wJgqyTs-gA', 'versions': None, 'checkpoint_reused': None}


<details>
<summary>💬 Discuss</summary>

- Every API so far answered with the data. Why might a service answer with a ticket instead?
- We sent an address, not a file. When would you have to upload the file instead?

</details>

## Step 2 — Wait for the result (10 minutes)

Ask the ticket's address whether the job is done. The [result check](https://documentation.datalab.to/api-reference/convert-result-check) answers with a `status`: `processing`, until it's `complete`. Asking again and again until a job is done is called **polling**. Take it in two moves: ask once by hand and print the status; then a given loop asks every two seconds until the job is complete, the way week 12's loop followed the `cursor` until there wasn't one. The finished answer says how many pages it read, `page_count`, and what they cost, `total_cost`, in cents.

In [5]:
# check: the JSON of a GET to job["request_check_url"] with the same headers; print its "status"
check = requests.get(job["request_check_url"], headers={"X-API-Key": datalab_key, **HEADERS}).json()
print(check["status"])

processing


In [6]:
# Given: ask again every two seconds until the job is complete; then print what it read and what it cost
while check["status"] != "complete":
    time.sleep(2)
    check = requests.get(job["request_check_url"], headers={"X-API-Key": datalab_key, **HEADERS}).json()
print(check["page_count"], "pages,", check["total_cost"], "cents,", round(check["runtime"], 1), "seconds of processing")

15 pages, 4.5 cents, 9.8 seconds of processing


<details>
<summary>💬 Discuss</summary>

- The loop has no limit. If Datalab never finished the job, what would your notebook do?
- Fifteen pages cost a few cents. About what would every year-end report Boulder has published cost?
- When would you choose `lite` mode, which cost the chapter about an eighth as much for this report?

</details>

## Step 3 — Read the Markdown, page by page (12 minutes)

The document is in `check["markdown"]`, one long string. With `paginate`, each page starts with a **page marker**: a number in braces, such as `{14}`, then a line of dashes. In the pattern we split on, `\{\d+\}` matches the number in its braces and `-{10,}` ten or more dashes. So [`re.split()`](https://docs.python.org/3/library/re.html#re.split) on that pattern gives a list of pages; drop the empty piece before the first marker. Python counts from 0, so page 2 is `pages[1]`. Print page 2, the summary table week 9 read with pdfplumber; then page 15, Exhibit 3, the picture of a table. **Don't print page 1:** the cover gives a staff member's email and phone number, and we keep contact details out of our outputs, as week 9 blacked them out. For the same reason, print Exhibit 3 from its table, which starts at the first `|`, below a note with the sales tax office's phone and email. Its lines are long, so print the first 150 characters of each. See the chapter's ["Reading the Markdown"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#reading-the-markdown).

In [7]:
# pages: re.split(r"\n*\{\d+\}-{10,}\n*", markdown) without its first, empty piece; print how many pages
# Print the first 1,200 characters of pages[1], the summary; then of pages[14], Exhibit 3. Never pages[0]
markdown = check["markdown"]
pages = re.split(r"\n*\{\d+\}-{10,}\n*", markdown)[1:]
print(len(pages), "pages")
print(pages[1][:1200])

15 pages
This report provides information and analysis related to 2024 unaudited taxes and fees including sales and use tax, accommodations tax, admissions tax, disposable bag fee, electronic smoking device tax, food service tax, marijuana tax, short-term rental tax, sugar sweetened beverage tax, and the trash tax. The revenue reported below was received by the city through January 2025 and is largely attributed to economic activity through the month of December. Details by industry, geographic zone and month are found on pages 3 through 7 of this report.

## **Sales and Use Tax Summary**

| <b>Sources - Related to Current Economic Activity</b>               | <b>December YTD</b>  |                      |                     |                     |                          |
|---------------------------------------------------------------------|----------------------|----------------------|---------------------|---------------------|--------------------------|
|                        

In [8]:
# exhibit: pages[14] from its first "|" (the table, below the note); print the first 150 characters of its first 10 lines
exhibit = pages[14][pages[14].index("|"):]
for line in exhibit.splitlines()[:10]:
    print(line[:150])

| REVENUE<br>CATEGORY            | YEAR        | JAN               | FEB               | MAR               | APR               | MAY               | J
|--------------------------------|-------------|-------------------|-------------------|-------------------|-------------------|-------------------|--
|                                |             |                   |                   |                   |                   |                   |  
| <b>Retail Sales Tax</b>        | 2019        | 8,590,452         | 7,489,624         | 9,606,398         | 8,214,047         | 8,832,480         | 1
|                                | 2020        | 7,761,028         | 7,370,943         | 10,025,017        | 6,090,136         | 7,059,371         | 9
|                                | 2021        | 8,059,343         | 7,608,759         | 10,351,245        | 8,666,637         | 9,229,065         | 1
|                                | 2022        | 9,592,143         | 9,009,882         | 11,37

### ⏱ Microsprint (4 minutes, alone or in pairs)

Datalab doesn't only read text: where a page has a chart, it writes a description of it. Chart 8, on page 9, is a pie chart of marijuana revenue, with its figures printed on the slices.

**Do:** print page 9 (`pages[8]`), and compare the description under the chart with the figures in its table.
**Share:** one sentence of the description that the figures contradict, and who wrote that sentence: the city, or Datalab?

In [9]:
# Microsprint: print pages[8]; read the description under Chart 8 against its figures
print(pages[8][:2200])

### Marijuana Tax Revenues

The City of Boulder collects revenue from the sale of both medical and recreational marijuana. The city collects the base 3.86% sales and use tax on medical and recreational marijuana. The city also collects an additional 3.50% sales and use tax on recreational marijuana, and a 5.0% excise tax when marijuana is transferred from a cultivation facility to a dispensary, testing facility or facility that produces marijuana infused products. In addition, the city receives a revenue share back from state recreational marijuana taxes, which is reflected in Chart 8 below.

**Chart 8: Distribution of YTD Marijuana Revenues**

![Pie chart: Breakdown of marijuana revenue by category and amount.](bedcca5cdf168e3508ef511d94ec514c_img.jpg)

| Category                                                | Amount      |
|---------------------------------------------------------|-------------|
| State Share-Back                                        | \$529,220   |
| Medical Mar

<details>
<summary>💬 Discuss</summary>

- The city's note asks readers to phone for a screen readable version of Exhibit 3. Who would have to check Datalab's table before the city could publish it as that version?
- The description of Chart 8 calls one slice the largest, then gives a larger one. How would you tell what a model wrote on these pages from what it read off them?
- Page 10's bar chart of sugar-sweetened beverage tax has no figures printed on it. Print `pages[9]`: where could the numbers Datalab wrote beside it have come from?

</details>

## Step 4 — Check what it read (12 minutes)

Week 9's first check was a sum: a row's months should add up to its total ([chapter 9, "Checking an Extraction"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#checking-an-extraction)). Exhibit 3 has 30 rows of figures, one per tax and year. In the Markdown, each row of the table is one line, with its cells between `|` characters. Split each line on `|`, and strip the `<b>` tags and the commas from its cells. Where `cells[2]` is a year, add the twelve months and compare the sum with the total. The report rounds each figure to the dollar, so allow $5, as week 9 did.

In [10]:
# For each line of pages[14]: cells = its pieces split on "|", stripped, without "<b>", "</b>" and ","
# Keep a line whose cells[2] is a year (re.fullmatch(r"20\d\d", ...)); add the 12 months, cells[3:15], and compare with cells[15]
# Count the rows checked, and print any whose months miss the total by more than $5
checked = 0
for line in pages[14].splitlines():
    cells = [c.strip().replace("<b>", "").replace("</b>", "").replace(",", "") for c in line.split("|")]
    if len(cells) > 16 and re.fullmatch(r"20\d\d", cells[2]):
        months = sum(int(c) for c in cells[3:15])
        checked += 1
        if abs(months - int(cells[15])) > 5:
            print(cells[2], months, cells[15])
print(checked, "rows checked")

30 rows checked


### ⏱ Microsprint (5 minutes, alone or in pairs)

The 2018 report is older: its Exhibit 3 is a table of text, not a picture.

**Do:** submit `revenue-report-2018.pdf` the same way (Steps 1 and 2), and run Step 4's check on its page 15, `pages_2018[14]`.
**Share:** its page count and cost, how many rows the check found, and how many missed.

In [11]:
# Microsprint: set data["file_url"] to the 2018 report; submit, wait (the loop again), split into pages_2018, and check pages_2018[14]
data["file_url"] = book + "revenue-report-2018.pdf"
job = requests.post(f"{datalab_base}/convert", data=data, headers={"X-API-Key": datalab_key, **HEADERS}).json()
check = requests.get(job["request_check_url"], headers={"X-API-Key": datalab_key, **HEADERS}).json()
while check["status"] != "complete":
    time.sleep(2)
    check = requests.get(job["request_check_url"], headers={"X-API-Key": datalab_key, **HEADERS}).json()
print(check["page_count"], "pages,", check["total_cost"], "cents")
pages_2018 = re.split(r"\n*\{\d+\}-{10,}\n*", check["markdown"])[1:]
checked = 0
for line in pages_2018[14].splitlines():
    cells = [c.strip().replace("<b>", "").replace("</b>", "").replace(",", "") for c in line.split("|")]
    if len(cells) > 16 and re.fullmatch(r"20\d\d", cells[2]):
        checked += 1
        if abs(sum(int(c) for c in cells[3:15]) - int(cells[15])) > 5:
            print(cells[2])
print(checked, "rows checked")

15 pages, 4.5 cents
28 rows checked


<details>
<summary>💬 Discuss</summary>

- Every row added up. What can a sum not catch? (Week 9's Docling put the right figures beside the wrong label.)
- Week 9 ran Docling on a laptop, free, in about a minute; Datalab took seconds and cost cents. What else differs between the two if the PDF holds a student's records rather than a city's?
- Our code assumes every request worked. If Datalab answered 401 (Unauthorized) because the key was wrong, which line would fail first?

</details>

## Step 5 — Save the text for Wednesday (3 minutes)

Write the 2024 report's Markdown to a file, so Wednesday doesn't have to pay to read it again. (Wednesday also works from the book's saved copies of all seven year-end reports, made the same way on 2026-11-07, so a student who missed today can start there.)

In [12]:
# Write markdown, the 2024 report's, to "revenue-report-2024-12.md" with open(..., "w", encoding="utf-8")
with open("revenue-report-2024-12.md", "w", encoding="utf-8") as f:
    f.write(markdown)

## End of Part 1

On Wednesday, Part 2 hands one page of this Markdown to a language model and asks for JSON in a shape we define: the summary table's totals. Then we check the model's numbers against week 9's, for seven years of reports. It starts with its own setup step, so you don't need today's cells to have run.

# Part 2 — Wednesday: a model as a function

A **language model**, the kind of program behind a chatbot, can be called like any other API: we send it an instruction and one page, and it returns an answer. We won't chat with it. We ask for **structured output**, JSON in a shape we define, so the answer goes straight into a DataFrame. Then we check it, because a model's numbers look right whether or not they are.

## Step 6 — Set up again (5 minutes)

Part 2 starts fresh, so a student who missed Monday can start here. Import what we need, check and read `OPENROUTER_KEY`, and set two addresses: OpenRouter's API, and the book's saved copies of Datalab's Markdown for the seven year-end reports (`data/ch-13/`), made as on Monday. [OpenRouter](https://openrouter.ai/docs/quickstart) takes one kind of request and passes it to models from many companies. Models whose ID ends in `:free` cost nothing, but they're shared, so they're sometimes busy.

In [13]:
# Import os, re, json, time, requests, and pandas (as pd); define HEADERS; check and read OPENROUTER_KEY
# openrouter_base = "https://openrouter.ai/api/v1"; saved = the book's data/ch-13/ address, as book was Monday's data/ch-09/
import os
import re
import json
import time
import requests
import pandas as pd

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
print("OPENROUTER_KEY:", "set" if os.environ.get("OPENROUTER_KEY") else "MISSING")
openrouter_key = os.environ["OPENROUTER_KEY"]
openrouter_base = "https://openrouter.ai/api/v1"
saved = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-13/"

OPENROUTER_KEY: set


In [14]:
# Given: only for a key the check said was MISSING, ask for it with getpass. If both said set, this does nothing
from getpass import getpass
for name in ["OPENROUTER_KEY"]:
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name} isn't set. Paste it here: ")
openrouter_key = os.environ["OPENROUTER_KEY"]

<details>
<summary>💬 Discuss</summary>

- OpenRouter stands between you and the companies that run the models. What does that make easier?
- With OpenRouter in the middle, who sees each page you send?

</details>

## Step 7 — Find the summary page (5 minutes)

Read the 2024 report's Markdown from the saved copy, split it into pages as on Monday, and keep the first page that holds the table's last row, `Total Sales and Use Tax`. That one page is all the model will see. It's about 1,500 **tokens**. A token is a word or a piece of one: the unit a model reads and writes text in, and is billed by.

In [15]:
# markdown: the .text of a GET to saved + "revenue-report-2024-12.md"; pages: split as on Monday
# summary: the first page that contains "Total Sales and Use Tax"; print its first 1,000 characters
markdown = requests.get(saved + "revenue-report-2024-12.md", headers=HEADERS).text
pages = re.split(r"\n*\{\d+\}-{10,}\n*", markdown)[1:]
summary = [page for page in pages if "Total Sales and Use Tax" in page][0]
print(summary[:1000])

This report provides information and analysis related to 2024 unaudited taxes and fees including sales and use tax, accommodations tax, admissions tax, disposable bag fee, electronic smoking device tax, food service tax, marijuana tax, short-term rental tax, sugar sweetened beverage tax, and the trash tax. The revenue reported below was received by the city through January 2025 and is largely attributed to economic activity through the month of December. Details by industry, geographic zone and month are found on pages 3 through 7 of this report.

## **Sales and Use Tax Summary**

| <b>Sources - Related to Current Economic Activity</b>               | <b>December YTD</b>  |                      |                     |                     |                          |
|---------------------------------------------------------------------|----------------------|----------------------|---------------------|---------------------|--------------------------|
|                                 

<details>
<summary>💬 Discuss</summary>

- The phrase is on two pages of every report, and we took the first. What would the model see if that page held a different table?
- What would the model miss if the table ran onto the next page?

</details>

## Step 8 — One request, one JSON answer (15 minutes)

A request to [chat completions](https://openrouter.ai/docs/api/reference/overview) holds a list of `messages`: a `system` message with the instruction, then a `user` message with the page. That's the whole conversation; we never add a third message. `response_format` asks for [structured outputs](https://openrouter.ai/docs/guides/features/structured-outputs). With `type` `json_schema`, the model must answer in the shape of the given `SCHEMA`, a **JSON Schema**: a description, itself in JSON, of the fields an answer must have and the type of each. **Temperature** sets how much chance goes into each word the model picks; `temperature` 0 asks for its most likely answer. We send a **fallback list** of free `models` rather than one model: if the first is busy, OpenRouter tries the next ([model fallbacks](https://openrouter.ai/docs/guides/routing/model-fallbacks)). Our `MODELS` list holds three free models, in this order: `nvidia/nemotron-3.5-super-vl:free`, `google/gemma-4-31b-it:free`, and `dots-studio/dots-3-note-preview:free` (checked on 2026-11-07; free models come and go, so check [OpenRouter's list](https://openrouter.ai/models?max_price=0) before class). The answer's text is in `choices[0]["message"]["content"]`, a JSON string, and [`json.loads()`](https://docs.python.org/3/library/json.html#json.loads) turns it into a dictionary. The response's `usage` counts the tokens the request used. Among them are `reasoning_tokens`: text some models write to work an answer out before they give it, which isn't part of the answer. See the chapter's ["One Request, One JSON Object"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#one-request-one-json-object).

In [16]:
# Given: the shape of the answer, as a JSON Schema: four fields, each with a type and a description
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
SCHEMA = {
    "type": "object",
    "properties": {
        "month": {"type": "string", "enum": MONTHS,
                  "description": "The month the year-to-date figures run through"},
        "year": {"type": "integer",
                 "description": "The report's current year: the later of the two years the table compares"},
        "total_prior": {"type": "integer",
                        "description": "The table's last row, Total Sales and Use Tax, for the earlier year, in dollars"},
        "total_current": {"type": "integer",
                          "description": "The table's last row, Total Sales and Use Tax, for the report's current year, in dollars"},
    },
    "required": ["month", "year", "total_prior", "total_current"],
    "additionalProperties": False,
}
INSTRUCTION = ("You extract figures from one page of a City of Boulder revenue report. "
               "Copy numbers exactly as printed, without $ or commas. Use only the page.")

In [17]:
# MODELS: three free models, in order; body: models, temperature 0, messages (system INSTRUCTION, user summary),
# and response_format {"type": "json_schema", "json_schema": {"name": "totals", "strict": True, "schema": SCHEMA}}
# response: the JSON of a POST to f"{openrouter_base}/chat/completions" with json=body and an Authorization header
# Print the model that answered and its "usage"; answer: json.loads() of the first choice's message content; print it
MODELS = ["nvidia/nemotron-3.5-super-vl:free", "google/gemma-4-31b-it:free", "dots-studio/dots-3-note-preview:free"]
body = {"models": MODELS, "temperature": 0,
        "messages": [{"role": "system", "content": INSTRUCTION}, {"role": "user", "content": summary}],
        "response_format": {"type": "json_schema", "json_schema": {"name": "totals", "strict": True, "schema": SCHEMA}}}
response = requests.post(f"{openrouter_base}/chat/completions", json=body,
                         headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
print(response["model"])
print(response["usage"])
answer = json.loads(response["choices"][0]["message"]["content"])
print(answer)

nvidia/nemotron-3.5-super-vl:free
{'prompt_tokens': 1568, 'completion_tokens': 2466, 'total_tokens': 4034, 'cost': 0, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}, 'completion_tokens_details': {'reasoning_tokens': 2413, 'image_tokens': 0, 'audio_tokens': 0}}
{'month': 'December', 'year': 2025, 'total_prior': 175517129, 'total_current': 173908431}


<details>
<summary>💬 Discuss</summary>

- If a different model answered your neighbor, can you compare your two results?
- Your `usage` shows how many `reasoning_tokens` the model wrote before its answer. Why would they matter on a paid model?
- The free models answer 429, Too Many Requests, when they're busy. If all three did, which line of this cell would fail?

</details>

## Step 9 — Check it against week 9 (10 minutes)

Week 9 read the same table with pdfplumber, by position, and checked its sums, so we trust its figures. They're our **ground truth**: the answers we check others against. Compare each of the model's fields with them: the two totals, the year, and the month. A model that copies numbers well can still misread which year is which.

In [18]:
# Given: week 9's figures for the last row, Total Sales and Use Tax, read with pdfplumber (chapter 9, "The Time Series"):
# for each report, the year before and the report's own year
TRUTH = {
    "revenue-report-2018": (134538892, 140648710), "revenue-report-2019": (140609751, 141954650),
    "revenue-report-2020": (141954650, 135195413), "revenue-report-2021-12": (135195413, 152302653),
    "revenue-report-2022-12": (152302653, 169112615), "revenue-report-2023-12": (169112615, 175517129),
    "revenue-report-2024-12": (175517129, 173908431),
}

In [19]:
# prior, current: TRUTH["revenue-report-2024-12"]; print, for each total, the model's figure, week 9's, and whether they match
# Then print the model's year and month: are they 2024 and December?
prior, current = TRUTH["revenue-report-2024-12"]
print("prior:  ", answer["total_prior"], prior, answer["total_prior"] == prior)
print("current:", answer["total_current"], current, answer["total_current"] == current)
print(answer["year"], answer["month"])

prior:   175517129 175517129 True
current: 173908431 173908431 True
2025 December


### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** send exactly the same request, `body`, three more times, two seconds apart, and print each answer.
**Share:** did all three agree, with each other and with week 9? Which field changed, if any?

In [20]:
# Microsprint: send body again three times, sleeping 2 seconds between; print the model and the answer each time
for attempt in range(3):
    response = requests.post(f"{openrouter_base}/chat/completions", json=body,
                             headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
    print(response["model"], json.loads(response["choices"][0]["message"]["content"]))
    time.sleep(2)

nvidia/nemotron-3.5-super-vl:free {'month': 'December', 'year': 2025, 'total_prior': 175517129, 'total_current': 173908431}


nvidia/nemotron-3.5-super-vl:free {'month': 'December', 'year': 2025, 'total_prior': 175517129, 'total_current': 173908431}


nvidia/nemotron-3.5-super-vl:free {'month': 'December', 'year': 2025, 'total_prior': 175517129, 'total_current': 173908431}


<details>
<summary>💬 Discuss</summary>

- Our three answers agreed with each other, the wrong year included. Does agreement make an answer right?
- `temperature` 0 asks for the most likely answer. Why might your answers still differ from your neighbor's, or from one day to the next?
- If the model had been wrong and you hadn't had week 9's figures, how would you have found out?

</details>

## Step 10 — Seven reports (15 minutes)

Now the same request for every year-end report from 2018 to 2024. We'll send it seven times, so gather Step 8's lines into a function, `extract_totals(page)`, that takes a page and returns the answer dictionary and the model that gave it. Then loop over `TRUTH`'s reports. For each, read its Markdown, find its summary page as in Step 7, call `extract_totals()`, and compare the answer with week 9's. Pause two seconds between requests. See the chapter's ["Seven Reports"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#seven-reports).

In [21]:
# extract_totals(page): Step 8's body and request, with the page as the user message; return the answer and response["model"]
def extract_totals(page):
    body = {"models": MODELS, "temperature": 0,
            "messages": [{"role": "system", "content": INSTRUCTION}, {"role": "user", "content": page}],
            "response_format": {"type": "json_schema", "json_schema": {"name": "totals", "strict": True, "schema": SCHEMA}}}
    response = requests.post(f"{openrouter_base}/chat/completions", json=body,
                             headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
    return json.loads(response["choices"][0]["message"]["content"]), response["model"]

In [22]:
# rows: for each report in TRUTH, read its Markdown, find the summary page, extract_totals() it, and add the answer,
# the model, and whether each total matches week 9; sleep 2 seconds between reports. results: a DataFrame; print it
rows = []
for report, (prior, current) in TRUTH.items():
    pages = re.split(r"\n*\{\d+\}-{10,}\n*", requests.get(saved + report + ".md", headers=HEADERS).text)[1:]
    summary = [page for page in pages if "Total Sales and Use Tax" in page][0]
    answer, model = extract_totals(summary)
    rows.append({"report": report, "model": model, **answer,
                 "prior_ok": answer["total_prior"] == prior, "current_ok": answer["total_current"] == current})
    time.sleep(2)
results = pd.DataFrame(rows)
print(results.drop(columns="model").to_string())
print(results["model"].value_counts())

                   report     month  year  total_prior  total_current  prior_ok  current_ok
0     revenue-report-2018  December  2018    134538892      139331838      True       False
1     revenue-report-2019   January  2020    141954650      139548151     False       False
2     revenue-report-2020   January  2021    135195413      133478757     False       False
3  revenue-report-2021-12  December  2021    135195413      152302653      True        True
4  revenue-report-2022-12  December  2022    152302653      169112615      True        True
5  revenue-report-2023-12  December  2023    169112615      175517129      True        True
6  revenue-report-2024-12  December  2025    175517129      173908431      True        True
model
nvidia/nemotron-3.5-super-vl:free    7
Name: count, dtype: int64


<details>
<summary>💬 Discuss</summary>

- Where on the page did a wrong report's figures come from? (Print its summary page and read the table's last two rows.)
- The 2024 report's year came back as 2025, and a sentence on its page says 2025. Would you fix that in the instruction, the schema's description, or a check afterwards?
- The loop has no error handling. If the fifth request got a 429 from every model in the list, what would `rows` hold when the cell stopped?

</details>

## End of Part 2

On Friday, Part 3 leaves the revenue reports for Wikipedia. Embeddings measure how the opening paragraphs of five politicians' articles have changed since 2004, and find the versions that answer a question. A model answers from them, with the IDs of its sources, or says it can't. Then we look at how MCP and skills describe tools to a model. It starts with its own setup step.

# Part 3 — Friday: search by meaning, answer with sources, and tools

On Wednesday the model read one page, and we chose the page. With a hundred passages, we first need a way to find the ones worth reading. An **embedding** turns a passage into a list of numbers, so that passages about the same thing get similar numbers. A question becomes a list of numbers too, and the passages closest to it are the ones to read.

Boulder's reports are a poor test of that. Their pages are mostly tables, and an embedding captures what a passage is about, not which figure it prints: every year's report has a page on marijuana revenue, and to an embedding those pages look nearly alike. Wikipedia suits it better. Its articles are prose, they're rewritten as their subjects' lives change, and every version is kept in the article's history (week 10). So today we take the **lead**, the paragraphs above an article's first heading, of five articles: Barack Obama, Hillary Clinton, Donald Trump, Joe Biden, and Kamala Harris. We take each lead as it stood at midnight UTC on January 1 of every year from 2004 to 2026, and as it stands now. How far was each year's lead from today's, and when did it change most?

Then a model answers a question from the leads, as JSON with the versions it used, or null. Finding the passages is **retrieval**, and writing the answer from them is generation: together, **retrieval-augmented generation**, or **RAG**. Ours keeps the answer to one year and its sources. Last, a look at how MCP and skills describe tools to a model.

## Step 11 — Set up, and one old version by hand (8 minutes)

Part 3 starts fresh: import what we need, now with [NumPy](https://numpy.org/doc/stable/user/absolute_beginners.html), matplotlib, and BeautifulSoup (week 6), and read `OPENROUTER_KEY`. Wikimedia asks for week 10's User-Agent, with the library and its version, so `HEADERS` takes that form today. Then get one old version of a lead by hand, in two requests to the Action API. First, [`prop=revisions`](https://www.mediawiki.org/wiki/API:Revisions) with `rvstart` set to `2009-01-01T00:00:00Z`, `rvdir=older` and `rvlimit=1` returns one revision: the newest at or before that moment, the version a reader saw on New Year's Day. Its `revid` names that version. Second, [`action=parse`](https://www.mediawiki.org/wiki/API:Parsing_wikitext) with `oldid` set to that `revid`, `prop=text` and `section=0` returns the version's lead as HTML (section 0 is everything above the first heading; `formatversion=2` makes `text` a plain string). Wikipedia keeps the wikitext of old versions, not their HTML, so `parse` draws the 2009 wikitext with today's templates. Templates deleted since then come back as links to missing pages, named `Template:…`. Read each paragraph (`<p>`) with BeautifulSoup's [`get_text()`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#get-text), and look for those leftovers. Then take them out, as the chapter's `lead_text()` does: [`select()`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#css-selectors-through-the-css-property) finds, by CSS selector (week 5), the tables, footnote marks, notes, links to missing templates (`a.new[title^='Template:']`) and error messages (`.error`), and `decompose()` deletes each. Last, a given cell loads all the leads, collected and cleaned the same way on 2026-11-09, and we check that your 2009 lead is the same text as the saved one. See the chapter's ["Embeddings and Retrieval"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#embeddings-and-retrieval).

In [ ]:
# Import os, re, json, time, requests, numpy (as np), pandas (as pd), matplotlib.pyplot (as plt), and BeautifulSoup from bs4
# HEADERS: week 10's User-Agent, with your email, then "python-requests/" and requests.__version__; check and read OPENROUTER_KEY
# openrouter_base as on Wednesday; API: Wikipedia's Action API, "https://en.wikipedia.org/w/api.php"
# wiki = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-13/data/wikipedia/"
import os
import re
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": f"WebDataScience/1.0 (your-email@colorado.edu) python-requests/{requests.__version__}"}
print("OPENROUTER_KEY:", "set" if os.environ.get("OPENROUTER_KEY") else "MISSING")
openrouter_key = os.environ["OPENROUTER_KEY"]
openrouter_base = "https://openrouter.ai/api/v1"
API = "https://en.wikipedia.org/w/api.php"
wiki = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-13/data/wikipedia/"

In [ ]:
# Given: only for a key the check said was MISSING, ask for it with getpass. If both said set, this does nothing
from getpass import getpass
for name in ["OPENROUTER_KEY"]:
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name} isn't set. Paste it here: ")
openrouter_key = os.environ["OPENROUTER_KEY"]

In [ ]:
# params: action=query, titles="Barack Obama", prop=revisions, rvprop="ids|timestamp", rvlimit=1,
#   rvstart="2009-01-01T00:00:00Z", rvdir="older", format=json
# Send it with HEADERS, go down to the page as in week 10, and print its first revision
params = {"action": "query", "titles": "Barack Obama", "prop": "revisions", "rvprop": "ids|timestamp",
          "rvlimit": 1, "rvstart": "2009-01-01T00:00:00Z", "rvdir": "older", "format": "json"}
data = requests.get(API, params=params, headers=HEADERS).json()
page = list(data["query"]["pages"].values())[0]
revision = page["revisions"][0]
print(revision)

In [ ]:
# params: action=parse, oldid=revision["revid"], prop=text, section=0, format=json, formatversion=2; html: the JSON's ["parse"]["text"]
# soup: BeautifulSoup(html, "html.parser"); print the first 150 characters of each <p>'s get_text(" ", strip=True)
params = {"action": "parse", "oldid": revision["revid"], "prop": "text", "section": 0,
          "format": "json", "formatversion": 2}
html = requests.get(API, params=params, headers=HEADERS).json()["parse"]["text"]
soup = BeautifulSoup(html, "html.parser")
for paragraph in soup.find_all("p"):
    print(paragraph.get_text(" ", strip=True)[:150])

In [ ]:
# For each tag soup.select() finds with "table, .reference, .hatnote, .shortdescription, style, a.new[title^='Template:'], .error",
# decompose() it. lead: the text of each <p> that has any, joined with blank lines, with re.sub(r" ([,.;:])", r"\1", ...)
# taking out the space get_text(" ") leaves before punctuation; print its first 300 characters
for tag in soup.select("table, .reference, .hatnote, .shortdescription, style, a.new[title^='Template:'], .error"):
    tag.decompose()
paragraphs = []
for paragraph in soup.find_all("p"):
    if paragraph.get_text(strip=True):
        paragraphs.append(paragraph.get_text(" ", strip=True))
lead = re.sub(r" ([,.;:])", r"\1", "\n\n".join(paragraphs))
print(lead[:300])

In [ ]:
# Given: every lead, collected and cleaned this way on 2026-11-09: one row per article and year, plus each
# article's "current" version. Collecting them at one request a second drew 429s for about 30 minutes
leads = pd.read_csv(wiki + "leads.csv")
print(len(leads), "leads")
print(leads.groupby("title")["year"].count())

In [ ]:
# saved: the "lead" of leads' row for "Barack Obama" and year "2009"; print whether lead == saved
saved = leads[(leads["title"] == "Barack Obama") & (leads["year"] == "2009")]["lead"].iloc[0]
print(lead == saved)

<details>
<summary>💬 Discuss</summary>

- Collecting these leads at one request a second drew 429 (Too Many Requests) for about 30 minutes, with waits of up to 40 seconds. If all of us ran that loop at once from this room, whose requests would Wikipedia refuse, and why does week 10's User-Agent matter here?
- Our two requests assume they worked. Wikimedia's 429 is plain text, not JSON (week 10): which line would fail, and what would it say?
- `parse` drew the 2009 version with today's templates, and our cleanup took out what it could see was broken. What might still come from today rather than from 2009, without looking broken?

</details>

## Step 12 — Embed, and compare each year with today (10 minutes)

[OpenRouter's embeddings endpoint](https://openrouter.ai/docs/api/reference/embeddings) takes a `model` and an `input`, a list of texts. It returns one **vector** per text: the list of numbers that is the text's embedding. We use one free model, `EMBEDDER`, for every vector. Two models turn the same text into different numbers, so their vectors can't be compared, and an embedding model never gets a fallback list. It reads at most about 4,096 tokens of a text, which is why we embed the lead and not the article: on January 1, 2009, Barack Obama's whole article was already 135,666 bytes of wikitext. Embed one lead by hand first, and count its numbers. Then the given cell embeds all 116 leads, 20 to a request, and scales each vector to length 1. (A vector's length here is the square root of its squared numbers added up, not how many numbers it has.) If the free model's daily limit stops that cell, the next one loads the same vectors, saved on 2026-11-09.

To compare two leads we use **cosine similarity**, which measures how nearly two vectors point the same way: 1 if they point exactly the same way, less the further apart they point. For vectors of length 1, it's their **dot product**, `@`: multiply the two lists number by number, and add up. Compare each lead with its article's current lead. Then [`pivot()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html) the result into a table with a row per year and a column per article, and plot it. See the chapter's ["Embedding and Cosine Similarity"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#embedding-and-cosine-similarity).

In [ ]:
# EMBEDDER = "nvidia/nemotron-3-embed-1b:free"; one: the JSON of a POST to f"{openrouter_base}/embeddings"
# with json {"model": EMBEDDER, "input": [leads["lead"][0]]}; print how many numbers its first embedding holds, and its "usage"
EMBEDDER = "nvidia/nemotron-3-embed-1b:free"
one = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": [leads["lead"][0]]},
                    headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
print(len(one["data"][0]["embedding"]), "numbers")
print(one["usage"])

In [ ]:
# Given: embed every lead, 20 to a request, with a pause between requests; vectors: one row per lead, scaled to length 1
vectors = []
for start in range(0, len(leads), 20):
    batch = leads["lead"][start:start + 20].tolist()
    response = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": batch},
                             headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
    vectors += [item["embedding"] for item in response["data"]]
    time.sleep(1)
vectors = np.array(vectors)
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
print(vectors.shape)

In [ ]:
# Given, only if the cell above stopped: remove the "#" from the next two lines, and run this cell. It loads the
# vectors saved on 2026-11-09, from the same model, already scaled to length 1, in the same order as leads
# import io
# vectors = np.load(io.BytesIO(requests.get(wiki + "lead-vectors.npy", headers=HEADERS).content))

In [ ]:
# today: a dictionary from each title to the vector of its "current" lead (row i of vectors is row i of leads)
# leads["similarity"]: each lead's vector @ today[its title]; print the current rows' similarity, which should be 1
today = {}
for i, row in leads.iterrows():
    if row["year"] == "current":
        today[row["title"]] = vectors[i]
leads["similarity"] = [vectors[i] @ today[title] for i, title in enumerate(leads["title"])]
print(leads[leads["year"] == "current"][["title", "similarity"]])

In [ ]:
# table: the leads whose year isn't "current", pivoted to a row per year and a column per title, with similarity as values
# Make its index (the years) integers; print it, rounded to 3 places
table = leads[leads["year"] != "current"].pivot(index="year", columns="title", values="similarity")
table.index = table.index.astype(int)
print(table.round(3))

In [ ]:
# Plot table with plt.subplots(): a line per article, with a marker at each year (marker="o")
# Give it a title, label both axes, and title the legend
fig, ax = plt.subplots(figsize=(8, 4.5))
table.plot(ax=ax, marker="o")
ax.set_title("How close each January 1 lead is to the article's current lead")
ax.set_xlabel("Year (the lead as of January 1, 00:00 UTC)")
ax.set_ylabel("Cosine similarity with the current lead")
ax.legend(title="Article")
plt.show()

### ⏱ Microsprint (4 minutes, alone or in pairs)

Each line climbs toward 1, but not evenly. A jump is a year whose lead moved much closer to today's than the year before's had.

**Do:** pick one article. [`diff()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.diff.html) gives each year's change from the year before, and `idxmax()` the year of the biggest rise. Print that year and the size of its jump; then print the first 400 characters of that year's lead and of the year before's.
**Share:** the article, the year, the jump, and what the lead said differently.

In [ ]:
# Microsprint: title: your article; jumps: table[title].diff(); year: jumps.idxmax(); print year and the jump, rounded to 3
# Then for the year before and that year, print the first 400 characters of the lead (leads' "year" holds text: use str())
title = "Kamala Harris"
jumps = table[title].diff()
year = jumps.idxmax()
print(year, round(jumps.max(), 3))
for y in [year - 1, year]:
    print(y, leads[(leads["title"] == title) & (leads["year"] == str(y))]["lead"].iloc[0][:400], "\n")

<details>
<summary>💬 Discuss</summary>

- Harris's lead jumped in 2021, when it first called her vice president-elect, and Biden's in 2009, for the same reason. Trump's biggest jump is 2009, not 2017: his lead grew from 93 words to 353 (the `words` column). What besides events can move a lead closer to today's?
- Clinton's January 2025 lead scores 0.992 against today's, higher than her January 2026 lead. What does a similarity measure, and which claims about how Wikipedia described these people can it not support?
- The given loop assumes every request worked. If the free model answered 429 to the fourth batch, the cell would stop at `response["data"]`: what would `vectors` hold, and what does the saved vectors' cell spare you?

</details>

## Step 13 — Answer with sources, as JSON (8 minutes)

Now the generation half of RAG. Our question is about when: *In which year did the article first say Joe Biden is the 46th president of the United States?* We'll ask more than one question, so first gather Step 12's lines into a function, `search(question, title)`. It embeds the question the way we embedded a lead, scales its vector to length 1, scores every lead against it with `@`, and returns the three best-scoring leads of one article. Those are the **passages**, each headed by its ID, such as `Joe Biden 2024`. Send them with the question and a schema whose `year` may be **null**, JSON's word for "no value" (`None` in Python): a null is the model's way to say the passages don't say. The request is Wednesday's, with a new instruction and schema. Then check the answer the way week 9 checked pdfplumber's: against the source, by reading the leads. See the chapter's ["Retrieval-Augmented Extraction"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#retrieval-augmented-extraction).

In [ ]:
# Given: each lead's ID, such as "Joe Biden 2024"; and the shape of an answer with its sources. year may be null
leads["id"] = leads["title"] + " " + leads["year"]
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "year": {"type": ["integer", "null"],
                 "description": "The earliest year whose passage states the fact; null if no passage states it"},
        "source_ids": {"type": "array", "items": {"type": "string"},
                       "description": "The IDs of the passages that state it; empty if year is null"},
    },
    "required": ["year", "source_ids"],
    "additionalProperties": False,
}
ANSWER_INSTRUCTION = ("Answer the question using only the passages, each headed by its ID. "
                      "A passage is one year's version of a Wikipedia article's lead. "
                      "Give a year only if a passage states the fact; if none does, answer null. "
                      "Answer with JSON that fits this schema: " + json.dumps(ANSWER_SCHEMA))

In [ ]:
# search(question, title): Step 12's lines, gathered: embed the question, scale its vector to length 1,
# put vectors @ it in leads["score"], and return the three best-scoring leads of that title
def search(question, title):
    response = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": [question]},
                             headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
    query = np.array(response["data"][0]["embedding"])
    leads["score"] = vectors @ (query / np.linalg.norm(query))
    return leads[leads["title"] == title].nlargest(3, "score")

In [ ]:
# question: "In which year did the article first say Joe Biden is the 46th president of the United States?"
# top: search(question, "Joe Biden"); print its IDs and scores; passages: each "ID: lead", joined with blank lines
question = "In which year did the article first say Joe Biden is the 46th president of the United States?"
top = search(question, "Joe Biden")
print(top[["id", "score"]])
passages = "\n\n".join(f"{row.id}: {row.lead}" for row in top.itertuples())

In [ ]:
# body: Wednesday's, with ANSWER_INSTRUCTION, the user message f"Question: {question}\n\n{passages}", and ANSWER_SCHEMA named "answer"
# Send it; print the model that answered and its answer
MODELS = ["nvidia/nemotron-3.5-super-vl:free", "google/gemma-4-31b-it:free", "dots-studio/dots-3-note-preview:free"]
body = {"models": MODELS, "temperature": 0,
        "messages": [{"role": "system", "content": ANSWER_INSTRUCTION},
                     {"role": "user", "content": f"Question: {question}\n\n{passages}"}],
        "response_format": {"type": "json_schema", "json_schema": {"name": "answer", "strict": True, "schema": ANSWER_SCHEMA}}}
response = requests.post(f"{openrouter_base}/chat/completions", json=body,
                         headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
print(response["model"], json.loads(response["choices"][0]["message"]["content"]))

In [ ]:
# Check it by reading: for each Joe Biden lead from 2020 on (leads["year"] >= "2020" keeps "current" too),
# print its ID and its first 300 characters. In which year's lead is he the 46th president, not about to be?
for row in leads[(leads["title"] == "Joe Biden") & (leads["year"] >= "2020")].itertuples():
    print(row.id, "|", row.lead[:300], "\n")

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** ask "In which year did the article first say Kamala Harris is the 47th president of the United States?" the same way: `search()` Kamala Harris's leads, and send the three best in `body`.
**Share:** the three IDs, and the model's answer. Is it right, and how do you know?

In [ ]:
# Microsprint: set question; top: search(question, "Kamala Harris"); print its IDs; rebuild passages,
# put them in body's user message (body["messages"][1]["content"]), and send it; print the model and its answer
question = "In which year did the article first say Kamala Harris is the 47th president of the United States?"
top = search(question, "Kamala Harris")
print(top[["id", "score"]])
passages = "\n\n".join(f"{row.id}: {row.lead}" for row in top.itertuples())
body["messages"][1]["content"] = f"Question: {question}\n\n{passages}"
response = requests.post(f"{openrouter_base}/chat/completions", json=body,
                         headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
print(response["model"], json.loads(response["choices"][0]["message"]["content"]))

<details>
<summary>💬 Discuss</summary>

- PLACEHOLDER, written from the run

</details>

## Step 14 — Tools: MCP and skills (6 minutes)

Some programs let a model call **tools**, such as a search or a database. The **Model Context Protocol** (MCP) is a shared way for a server to say which tools it offers. You ask it with **JSON-RPC**: a request in JSON that names a `method` for the server to run, with an `id` that its answer repeats. The method `tools/list` returns each tool's name, its description, and an `inputSchema`, a JSON Schema like Wednesday's ([the specification](https://modelcontextprotocol.io/specification/2025-06-18/server/tools)). DeepWiki runs a public MCP server that needs no key. Send it `initialize`, then `tools/list`. It answers as a short **event stream**: lines of text, each starting with a label such as `event:` or `data:`, so the JSON is on the line that starts with `data: `. A **skill** is plainer still: a folder of instructions in Markdown, whose `name` and `description` tell a model when to read the rest. Read the first lines of a public one, for PDFs, and its headings. See the chapter's ["Tools, MCP, and Skills"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#tools-mcp-and-skills).

In [35]:
# mcp: "https://mcp.deepwiki.com/mcp"; mcp_headers: HEADERS plus "Accept": "application/json, text/event-stream"
# hello: {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18",
#   "capabilities": {}, "clientInfo": {"name": "info4617", "version": "1.0"}}}; POST it; print the first 300 characters of the .text
mcp = "https://mcp.deepwiki.com/mcp"
mcp_headers = {**HEADERS, "Accept": "application/json, text/event-stream"}
hello = {"jsonrpc": "2.0", "id": 1, "method": "initialize",
         "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "info4617", "version": "1.0"}}}
print(requests.post(mcp, json=hello, headers=mcp_headers).text[:300])

event: message
data: {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-06-18","capabilities":{"logging":{},"prompts":{"listChanged":true},"resources":{"subscribe":false,"listChanged":true},"tools":{"listChanged":true}},"serverInfo":{"name":"DeepWiki","version":"4.0.2"},"instructions":"DeepW


In [36]:
# listing: the .text of a POST of {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}; line: its line that starts with "data: "
# For each tool in json.loads() of the line after "data: ", ["result"]["tools"], print its name and its required inputs
listing = requests.post(mcp, json={"jsonrpc": "2.0", "id": 2, "method": "tools/list"}, headers=mcp_headers).text
line = [l for l in listing.splitlines() if l.startswith("data: ")][0]
for tool in json.loads(line[6:])["result"]["tools"]:
    print(tool["name"], tool["inputSchema"]["required"])

ask_wiki_question ['repoName', 'question']
read_wiki_contents ['repoName']
read_wiki_structure ['repoName']


In [37]:
# skill: the .text of a GET to "https://raw.githubusercontent.com/anthropics/skills/main/skills/pdf/SKILL.md"; print its first 5 lines,
# then the headings that start with "### ", the tools it teaches
skill = requests.get("https://raw.githubusercontent.com/anthropics/skills/main/skills/pdf/SKILL.md", headers=HEADERS).text
print("\n".join(skill.splitlines()[:5]))
print([line for line in skill.splitlines() if line.startswith("### ")])

---
name: pdf
description: Use this skill whenever the user wants to do anything with PDF files. This includes reading or extracting text/tables from PDFs, combining or merging multiple PDFs into one, splitting PDFs apart, rotating pages, adding watermarks, creating new PDFs, filling PDF forms, encrypting/decrypting PDFs, extracting images, and OCR on scanned PDFs to make them searchable. If the user mentions a .pdf file or asks to produce one, use this skill.
license: Complete terms in LICENSE.txt
---
['### pypdf - Basic Operations', '### pdfplumber - Text and Table Extraction', '### reportlab - Create PDFs', '### pdftotext (poppler-utils)', '### qpdf', '### pdftk (if available)', '### Extract Text from Scanned PDFs', '### Add Watermark', '### Extract Images', '### Password Protection']


<details>
<summary>💬 Discuss</summary>

- An MCP tool's `inputSchema` and Wednesday's `response_format` are both JSON Schema. What does each one constrain: what the model may send, or what it must return?
- The PDF skill tells a model to use pypdf and pdfplumber, week 9's tools. If a model followed it on Boulder's reports, which of week 9's checks would still be yours to run?

</details>

## Step 15 — What did we learn? (3 minutes)

In a few sentences, as comments:

- Datalab read a picture of a table and wrote descriptions of charts. Which of its output would you publish, and which would you check first?
- On Wednesday, which reports did the model get wrong, and was the problem the model or what we sent it?
- Step 12 put a number on how far each year's lead was from today's. Which of its jumps could you explain only by reading the leads, and what can a similarity not tell you?
- Every call this week returned JSON, and every answer was checked: against week 9's figures, a sum, or the leads themselves. What would you check your own project's model outputs against?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, convert a public PDF you choose, with no private individuals' names: Monday's Steps 1–3.
- Step 2, a schema of your own: Wednesday's `SCHEMA`, with fields your document prints.
- Step 3, extract: Step 8's request, and Step 10's `extract_totals()` for more than one page.
- Step 4, check: Step 4's sums and Step 9's ground truth; find something to check against.
- Step 5, retrieve and answer with sources, including a question your document can't answer: Friday's Steps 12–13, the embeddings, `search()`, and a schema whose answer may be null. Our leads were short enough to embed whole; a document's pages may first need cutting into chunks, as the chapter shows.
- Step 6, report the costs: Step 2's `page_count` and `total_cost`, and Step 8's `usage`.

In [38]:
# Your answers here, as comments
# PLACEHOLDER

## Friday: your portfolio checkpoint (15 minutes)

The last 15 minutes go to your project. This week's checkpoint, **Analyze your data**, is due Sunday, November 15:

- **An analysis notebook** that runs top to bottom, with its output visible;
- **results that answer at least part of your question**: the counts or comparisons your proposal named;
- **at least one figure** saved in your repository, with a title, labeled axes, and a one-sentence caption;
- **the README's Methods section** and your preliminary **Findings**.

Use the time to run your analysis notebook once, top to bottom, here, where you can ask. If you use a model anywhere in your analysis, say which, when, and what you checked its output against, in Methods.